# 01f · 가역성과 MCMC: 세부균형과 메트로폴리스-헤이스팅스 (Reversibility and MCMC: Detailed Balance and Metropolis-Hastings)

| 항목 | 내용 |
|---|---|
| 교재 지도 | Durrett 3e §1.6 (세부균형 (1.11), Ehrenfest·그래프 위 랜덤 워크, Metropolis-Hastings) · Ross 12e §4.8 (시간 가역 사슬, Kolmogorov 순환 판정 Thm 4.2), §4.9 (MCMC, Gibbs sampler) · Norris §1.9 (시간 반전 Thm 1.9.1, 세부균형 ⇒ 불변 Lemma 1.9.2) · Lawler 2e Ch. 7 (가역 사슬의 스펙트럼, $\lambda_2$와 자기상관) |
| 선수 노트북 | 01e, 01d |
| 예상 소요 | 100분 |
| 상태 | draft |

01d에서 정상분포 $\pi$는 "총 흐름의 균형" $\sum_x\pi(x)p(x,y)=\pi(y)$로 정의했고, 01e에서 가역 사슬에 대해서만 스펙트럼 상계 $\|P^n(x,\cdot)-\pi\|_{TV}\le\frac{\lambda_*^n}{2\sqrt{\pi(x)}}$를 썼다. 이 노트북은 그 "가역"이 정확히 무엇인지 — **쌍마다의 흐름 균형(세부균형)** — 를 정의하고, 그 조건을 **설계 원리로 뒤집어** 원하는 $\pi$를 정상분포로 갖는 사슬을 만드는 **Metropolis-Hastings** 알고리즘을 세운다. 그리고 MCMC의 진짜 비용이 "수렴"이 아니라 **자기상관**(적분 자기상관 시간 $\tau$, 유효 표본 크기 $n/\tau$)임을 정확값과 시뮬레이션으로 확인한다.

## 0. 준비 (Setup)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from numpy.linalg import matrix_power
from scipy.stats import binom, norm
from IPython.display import Markdown
from spkit import rng_for, fast, scale, setup_plots
from spkit.mc import mc_estimate, mc_proportion, compare_table, assert_close
from spkit.markov import (is_stochastic, simulate_chain, stationary,
                          ehrenfest_matrix, metropolis_chain)
from spkit.plots import plot_paths

SEED, rng = rng_for("01f")
setup_plots()
N_PATHS = scale(20_000)      # FAST(SPKIT_FAST=1) 모드에서는 1/10 크기 (번인 실험 경로 수)
N_PAIRS = scale(100_000)     # 정상 출발 (X0, X1) 쌍의 수 (ρ1 추정)
N_LONG = scale(100_000)      # 경험 ACF 용 긴 사슬 한 개의 길이
N_CHAINS, N_STEPS, BURN_IN = 200, 1_000, 200   # 독립 사슬 실험 (FAST 와 같음; 순수 파이썬 루프 20만 스텝 ≈ 0.2초)
print(f"SEED={SEED}, fast={fast()}, N_PATHS={N_PATHS}, N_PAIRS={N_PAIRS}, N_LONG={N_LONG}")

## 1. Recall — 지난 노트북에서

책을 덮고 먼저 답을 적어 본 뒤 정답을 펼쳐 보세요.

**Q1.** (01e) 수렴 정리 $P^n(x,\cdot)\to\pi$의 가정 두 개는 무엇이고, 각 가정이 깨졌을 때 무슨 일이 생기는가?

<details><summary>정답</summary>

**기약** + **비주기** (유한, 또는 양재귀). 기약이 깨지면 극한이 출발 클래스에 의존하고, 비주기가 깨지면 $P^n$이 진동하며 세자로 평균 $\frac1n\sum_{k<n}P^k$만 수렴한다.

</details>

**Q2.** (01e) 가역 사슬에서 TV 거리의 스펙트럼 상계와 '스펙트럼 간극'의 정의는?

<details><summary>정답</summary>

$\|P^n(x,\cdot)-\pi\|_{TV}\le\dfrac{1}{2\sqrt{\pi(x)}}\lambda_*^n$, $\lambda_*=\max(|\lambda_2|,|\lambda_{\min}|)$; 간극은 $1-\lambda_*$. "가역"이라는 가정이 어디에 쓰였는지가 오늘의 정리 3이다.

</details>

**Q3.** (01e) 결합(coupling) 부등식을 쓰고, 수렴 정리 증명에서 비주기성이 어디에 쓰이는지 말하시오.

<details><summary>정답</summary>

$\|P^n(x,\cdot)-\pi\|_{TV}\le P(T_c>n)$ ($T_c$는 두 사슬이 만나는 시각). 비주기 + 기약 ⇒ 충분히 큰 $n$에서 모든 $x,y$에 $p^n(x,y)>0$ ⇒ 독립 곱 사슬 $(X_n,Y_n)$이 기약 ⇒ 대각선 $\{(z,z)\}$에 확률 1로 도달, 즉 $T_c<\infty$ a.s.

</details>

**Q4.** (01d) 이중확률행렬의 정상분포는? 공 $n$개 Ehrenfest 사슬의 정상분포는?

<details><summary>정답</summary>

균등분포 $1/|S|$; $\text{Bin}(n,1/2)$. Ehrenfest 증명(01d 정리 2)의 1단계가 바로 오늘의 세부균형이었다.

</details>

## 2. 이론 — 내 말로 다시 쓰기

### 2.1 정의

- **세부균형 (detailed balance)**: 확률벡터 $\pi$와 전이행렬 $P$가
  $$\pi(x)\,p(x,y)=\pi(y)\,p(y,x)\qquad\forall x,y$$
  를 만족한다. 흐름 $F(x,y)=\pi(x)p(x,y)$가 **쌍마다** 균형 — 정상성 $\sum_x\pi(x)p(x,y)=\pi(y)$(한 상태로 들어오는 **총** 흐름 = 나가는 총 흐름)보다 강한 조건이다. 행렬로는 $F=D_\pi P$가 대칭($F=F^\top$)이라는 말이다.
- **가역 (reversible)**: 세부균형을 만족하는 정상분포 $\pi$를 갖는 사슬. 정상 상태($X_0\sim\pi$)에서 $(X_0,\dots,X_n)$과 $(X_n,\dots,X_0)$의 법칙이 같다 — "필름을 거꾸로 돌려도 구별할 수 없다".
- **시간 반전 사슬 (time reversal)**: $\pi$가 정상분포일 때
  $$\hat p(x,y)=\frac{\pi(y)\,p(y,x)}{\pi(x)},\qquad \hat P=D_\pi^{-1}P^\top D_\pi.$$
  행합은 $\sum_y\pi(y)p(y,x)/\pi(x)=\pi(x)/\pi(x)=1$ — 여기서 **정상성**이 쓰인다. 정상 상태의 사슬을 거꾸로 돌리면 전이행렬 $\hat P$인 마르코프 사슬이다(Norris Thm 1.9.1): $P_\pi(X_{n}=y\mid X_{n+1}=x)=\pi(y)p(y,x)/\pi(x)$. 가역 $\iff\hat P=P$.
- **Metropolis-Hastings (MH)**: 목표분포 $\pi$(정규화 불필요)와 제안행렬 $q(x,y)$(행확률행렬)가 주어질 때, 현재 $x$에서 $y\sim q(x,\cdot)$를 제안하고 확률
  $$\alpha(x,y)=\min\!\Big(1,\ \frac{\pi(y)\,q(y,x)}{\pi(x)\,q(x,y)}\Big)$$
  로 **수락**(이동), 아니면 $x$에 **머문다**. 전이행렬은 $p(x,y)=q(x,y)\alpha(x,y)$ ($y\ne x$), $p(x,x)=1-\sum_{y\ne x}p(x,y)$. 대칭 제안 $q(x,y)=q(y,x)$이면 $\alpha=\min(1,\pi(y)/\pi(x))$ — 원래의 **Metropolis** 알고리즘. `metropolis_chain(target, proposal)`이 이 행렬을 만든다.
- **자기상관 함수 (autocorrelation)**: 정상 상태에서 관측 함수 $f$에 대해 $\bar f=f-E_\pi f$, $\sigma_f^2=\operatorname{Var}_\pi f$라 할 때
  $$\rho_k=\operatorname{Corr}\big(f(X_0),f(X_k)\big)=\frac{\sum_x\pi(x)\,\bar f(x)\,(P^k\bar f)(x)}{\sigma_f^2}.$$
  ($E[\bar f(X_0)\bar f(X_k)]=\sum_x\pi(x)\bar f(x)\sum_yp^k(x,y)\bar f(y)$.)
- **적분 자기상관 시간 (integrated autocorrelation time)** $\tau=1+2\sum_{k\ge1}\rho_k$. **유효 표본 크기** $n_{\rm eff}=n/\tau$: 길이 $n$ 사슬의 시간 평균 $\bar f_n$의 분산이 $\approx\sigma_f^2\tau/n$이므로, 독립 표본 $n/\tau$개와 같은 정보량이다.
- **번인 (burn-in)**: 초기 상태의 영향이 사라질 때까지 버리는 앞부분. 필요한 길이는 혼합시간 $t_{\rm mix}$ 규모, 즉 $\lambda_*^n$이 충분히 작아지는 $n$.

### 2.2 정리 1 — 세부균형 ⇒ 정상분포 (Durrett 3e §1.6 (1.11); Norris §1.9 Lemma 1.9.2)

> 확률벡터 $\pi$가 세부균형 $\pi(x)p(x,y)=\pi(y)p(y,x)$ $\forall x,y$를 만족하면 $\pi P=\pi$.

**가정이 왜 필요한가**
- **행합 1** ($\sum_xp(y,x)=1$): 증명의 마지막 등호에 쓰인다. 세부균형만으로는 $\pi(y)\sum_xp(y,x)$까지밖에 못 간다.
- **역은 성립하지 않는다**: 정상분포가 있어도 세부균형이 아닐 수 있다. 3-순환 편향 사슬(4.1절)은 정상분포가 균등이지만 시계방향으로 도는 순환 흐름이 있어 $F\ne F^\top$. 세부균형은 정상분포를 **찾는 지름길**이지 정상분포의 특징이 아니다.

<details><summary>증명</summary>

임의의 $y$에 대해
$$\sum_x\pi(x)\,p(x,y)\overset{\text{세부균형}}{=}\sum_x\pi(y)\,p(y,x)=\pi(y)\sum_xp(y,x)\overset{\text{행합 }1}{=}\pi(y).$$
즉 $(\pi P)(y)=\pi(y)$ $\forall y$. $\square$

*왜 지름길인가.* $\pi P=\pi$는 $|S|$개 미지수의 연립방정식이지만, 세부균형은 "이웃 $x\to y$마다 $\pi(y)=\pi(x)\,p(x,y)/p(y,x)$"라는 **국소 점화식**이다. 생사(birth–death) 사슬처럼 이웃끼리만 움직이는 사슬에서는 $\pi(k)=\pi(0)\prod_{i<k}\frac{p(i,i+1)}{p(i+1,i)}$로 곧바로 풀린다 — Ehrenfest의 $\text{Bin}(n,1/2)$가 그 예. 세부균형이 **실패하면**(순환 흐름) 이 점화식은 모순을 내고, 그때는 01d의 선형계로 돌아가야 한다.

</details>

### 2.3 정리 2 — Metropolis-Hastings 사슬의 세부균형 (Durrett 3e §1.6; Ross 12e §4.9)

> 2.1의 구성 $p(x,y)=q(x,y)\alpha(x,y)$는 $\pi$에 대해 세부균형을 만족한다. 따라서 $\pi$는 정상분포이고, 사슬이 기약·비주기이면 $P^n(x,\cdot)\to\pi$이며 시간 평균 $\frac1n\sum_{k=1}^nf(X_k)\to E_\pi f$.

**가정이 왜 필요한가**
- $q(x,y)>0\iff q(y,x)>0$: 비율 $q(y,x)/q(x,y)$가 정의되도록. 한쪽만 0이면 $\alpha=0$으로 두어 그 이동을 금지하면 되고(`metropolis_chain`은 `nan → 0`으로 처리), 세부균형은 $0=0$으로 유지된다.
- **기약성**: 제안이 $\pi$의 지지집합 전체를 연결해야 유일한 정상분포로 수렴한다. E3의 쌍봉 목표에서는 기약이긴 하지만 $\lambda_2\approx1$이라 "사실상" 깨진다.
- **비주기**: 거절(머묾) 확률 $p(x,x)>0$인 상태가 하나라도 있으면(기약 사슬에서) 자동으로 성립. 거절이 전혀 없는 특수한 경우(예: $\pi$ 균등 + 대칭 제안 = 제안 사슬 자체)에는 제안 사슬의 주기를 따로 봐야 한다.
- **$\pi$의 정규화 상수는 비율 $\pi(y)/\pi(x)$에서 소거된다**: MCMC의 핵심 장점. $\pi(x)=e^{-H(x)}/Z$에서 $Z=\sum_xe^{-H(x)}$를 계산할 수 없어도(상태가 $2^{1000}$개) 사슬은 돌릴 수 있다.

<details><summary>증명</summary>

$x\ne y$이고 $q(x,y)>0$인 쌍을 잡자. 일반성을 잃지 않고 $\pi(y)q(y,x)\le\pi(x)q(x,y)$라 하자. 그러면
$$\alpha(x,y)=\frac{\pi(y)q(y,x)}{\pi(x)q(x,y)},\qquad\alpha(y,x)=\min\!\Big(1,\frac{\pi(x)q(x,y)}{\pi(y)q(y,x)}\Big)=1.$$
따라서
$$\pi(x)\,p(x,y)=\pi(x)\,q(x,y)\,\alpha(x,y)=\pi(y)\,q(y,x)=\pi(y)\,q(y,x)\,\alpha(y,x)=\pi(y)\,p(y,x).$$
$q(x,y)=0$이면 $q(y,x)=0$이므로 양변이 0, $x=y$는 자명. 즉 세부균형이 성립하고, 정리 1에 의해 $\pi P=\pi$. 기약·비주기이면 01e 수렴 정리로 $P^n(x,\cdot)\to\pi$, 01d 에르고딕 정리로 시간 평균이 $E_\pi f$로 수렴한다. $\square$

*수락확률이 왜 이 모양인가.* "제안 흐름" $\pi(x)q(x,y)$와 $\pi(y)q(y,x)$가 다르면 큰 쪽을 작은 쪽에 맞춰 **깎는다**. 깎는 비율이 $\alpha$이고, 깎아서 못 간 확률은 $x$에 머무는 데 쓴다(머무는 것은 세부균형에 영향이 없다: $x=y$ 항). 이것이 세부균형을 맞추는 **최소한의 수정**이며, 그래서 한쪽의 $\alpha$는 항상 1이다.

</details>

### 2.4 정리 3 — 가역 사슬의 실수 스펙트럼 (Lawler 2e Ch. 7)

> 가역 사슬의 $P$는 실수 고유값 $1=\lambda_1\ge\lambda_2\ge\dots\ge\lambda_{|S|}\ge-1$을 가지며, $S=D_\pi^{1/2}PD_\pi^{-1/2}$는 대칭행렬이다.

**가정이 왜 필요한가**
- **세부균형**: $S_{xy}=\sqrt{\pi(x)/\pi(y)}\,p(x,y)=\dfrac{\pi(x)p(x,y)}{\sqrt{\pi(x)\pi(y)}}$의 분자가 $x\leftrightarrow y$ 대칭인 것이 정확히 세부균형이다. 가역이 아니면 실수 스펙트럼이 보장되지 않는다 — 3-순환 편향 사슬의 고유값은 $-\tfrac12\pm0.693i$ (4.1절).

**증명 스케치.** $S$가 대칭이므로 실수 고유값과 직교 고유벡터 $\{v_j\}$를 갖는다. $S=D^{1/2}PD^{-1/2}$는 $P$와 닮음이므로 고유값이 같고, $P$의 오른쪽 고유벡터는 $f_j=D^{-1/2}v_j$로, $\langle f_i,f_j\rangle_\pi=\sum_x\pi(x)f_i(x)f_j(x)=v_i^\top v_j=\delta_{ij}$ — $L^2(\pi)$ 직교 기저. $f_1\equiv1$ ($\lambda_1=1$). 이것이 01e 스펙트럼 상계의 전제이고, 아래 정리 4의 "스펙트럼 표현"의 재료다. $\square$

### 2.5 정리 4 — 시간 평균의 분산과 $\tau$ (Lawler 2e Ch. 7)

> 정상 상태에서
> $$\operatorname{Var}\Big(\frac1n\sum_{k=1}^nf(X_k)\Big)=\frac{\sigma_f^2}{n}\Big(1+2\sum_{k=1}^{n-1}\big(1-\tfrac kn\big)\rho_k\Big)=:\frac{\sigma_f^2\,\tau_n}{n}\ \xrightarrow[\ n\to\infty\ ]{}\ \frac{\sigma_f^2\,\tau}{n}\ (\text{비율}).$$
> 가역 사슬에서는 $\rho_k=\sum_{j\ge2}w_j\lambda_j^k$ ($w_j\ge0$, $\sum_jw_j=1$)이고 $\tau\le\dfrac{1+\lambda_2}{1-\lambda_2}$.

**가정이 왜 필요한가**
- **정상 출발** (또는 충분한 번인 뒤): 아니면 $E\bar f_n\ne E_\pi f$인 **편향** 항이 추가된다. 번인은 편향을, $\tau$는 분산을 다룬다.
- **가역성**: 스펙트럼 표현 $\rho_k=\sum_jw_j\lambda_j^k$가 정리 3의 직교 고유기저에서 나온다. 그러면 $1+2\sum_{k\ge1}\lambda^k=\frac{1+\lambda}{1-\lambda}$가 $\lambda$에 대해 증가하므로 $\tau=\sum_jw_j\frac{1+\lambda_j}{1-\lambda_j}\le\frac{1+\lambda_2}{1-\lambda_2}$.

**증명 스케치.** 분산을 전개하면 $\frac1{n^2}\sum_{j,k}\operatorname{Cov}(f(X_j),f(X_k))=\frac{\sigma_f^2}{n^2}\big(n+2\sum_{k=1}^{n-1}(n-k)\rho_k\big)$ (정상 상태라 공분산이 $|j-k|$에만 의존). $n\to\infty$에서 괄호$/n\to1+2\sum_k\rho_k=\tau$ (지배수렴; $|\rho_k|\le\lambda_*^k$). 스펙트럼: $\bar f=\sum_{j\ge2}a_jf_j$ (정리 3의 기저; $j=1$ 항은 $E_\pi\bar f=0$이라 없음), $P^k\bar f=\sum_ja_j\lambda_j^kf_j$, 직교성으로 $\rho_k=\sum_ja_j^2\lambda_j^k/\sum_ja_j^2$, 즉 $w_j=a_j^2/\sigma_f^2$. $\square$

### 2.6 직관과 함정

**직관.** 세부균형은 "필름을 거꾸로 돌려도 구별할 수 없다"는 뜻이다. $x\to y$ 흐름과 $y\to x$ 흐름이 쌍마다 같으니 어떤 순환 흐름도 없다. 3-순환 편향 사슬은 정상분포가 균등이지만 시계방향 순환 흐름이 있어 가역이 아니다 — 거꾸로 돌리면 반시계로 도는 다른 사슬 $\hat P=P^\top$이 보인다. MCMC는 이 조건을 **설계 원리로 뒤집는다**: 원하는 $\pi$를 정하고, 아무 제안이나 한 뒤 세부균형이 맞도록 수락확률을 깎으면 정상분포가 $\pi$인 사슬이 자동으로 나온다 — 정규화 상수를 몰라도. 값을 치르는 곳은 **속도**다: 사슬의 표본은 독립이 아니라 $\tau\approx20$걸음마다 하나꼴로만 새 정보를 주고($n_{\rm eff}=n/20$), $\lambda_2$가 1에 가까우면 $\tau$가 폭발한다(쌍봉 목표: $\lambda_2=0.9992$, $\tau$ 상계 2594).

**함정.**
- **정상분포 ≠ 가역.** 검사는 $F=\operatorname{diag}(\pi)P$가 대칭인지($F=F^\top$)로 한다. 이중확률행렬이면 균등이 정상이지만 $P$가 대칭이 아니면 가역이 아니다.
- **수락률은 높을수록 좋은 게 아니다.** $\pm1$ 제안은 수락률 81%지만 $\tau\approx20$; 균등 독립 제안은 수락률 27%지만 $\lambda_2=0.75$로 훨씬 빨리 섞인다(E2). 효율의 기준은 $\tau$(또는 $\lambda_2$)이지 수락률이 아니다.
- **MCMC 표본의 SE를 `std/sqrt(n)`으로 계산하면** $\sqrt\tau\approx4.5$배 과소평가된다. 배치 평균이나 독립 사슬 여러 개로 SE를 구할 것(4.4절).
- **번인을 아무리 길게 해도 표본 간 자기상관은 사라지지 않는다.** 번인은 편향을, $\tau$는 분산을 다룬다.
- **경계 밖 제안**(예: $x=0$에서 $-1$)은 **'거절'로 처리**해야 대칭 제안이 유지된다. 경계에서 반사시키거나 재추첨하면 $q$가 비대칭이 되어 수락확률에 $q$ 비율을 넣어야 한다.

✍️ 내 말로: (책을 덮고 여기에 핵심 정의·정리를 다시 써 보세요 — (1) 세부균형과 정상성의 차이, (2) MH 수락확률이 왜 그렇게 생겼는지, (3) $\tau$와 유효 표본 크기의 뜻을 각각 두 문장으로.)

## 3. Predict — 코드를 돌리기 전에 예측

목표분포는 $\pi=\text{Bin}(20,0.3)$ on $\{0,\dots,20\}$, 제안은 $\pm1$ 대칭(각 $1/2$, 경계 밖 제안은 거절), 관측 함수는 $f(x)=x$이다. 숫자를 먼저 적어 보세요 — 틀려도 됩니다. 정확값은 5절에서 대조합니다.

In [ ]:
predictions = {
    "mean_target": None,        # 목표분포 Bin(20,0.3)의 평균은?
    "var_target": None,         # 그 분산은?
    "lambda2_mh": None,         # ±1 대칭 제안 Metropolis 사슬 P_MH 의 두 번째 고유값 λ2 는? (힌트: 0.85~0.95)
    "accept_rate": None,        # 정상 상태에서 제안이 수락되는 비율 Σ_x π(x) Σ_{y≠x} q(x,y) min(1, π(y)/π(x)) 은?
    "rho1": None,               # f(x)=x 의 정상 상태 lag-1 자기상관 ρ1 은?
    "tau_int": None,            # 적분 자기상관 시간 τ = 1 + 2 Σ_{k≥1} ρ_k 는? (상계 (1+λ2)/(1-λ2) 와 비교해 보세요)
    "n_eff_1e5": None,          # 길이 10^5 사슬의 유효 표본 크기 n/τ 는?
    "mean_after_10_from_20": None,   # X0 = 20 (꼬리) 에서 출발한 MH 사슬의 E[X_10] 은?
    "mean_after_50_from_20": None,   # 같은 사슬의 E[X_50] 은? (번인 필요량의 감)
}

## 4. Simulate — 시뮬레이션

### 4.0 도우미 함수

`spkit`에 없는 다섯 개를 여기서 정의한다: $\pm1$ 제안행렬, 시간 반전, 정확 ACF(행렬 거듭제곱), 경험 ACF, 명시적 Metropolis 걷기.

In [ ]:
# helper (spkit에 없어서 여기서 정의)
def proposal_pm1(K):
    """K×K 대칭 ±1 제안행렬. 경계 밖 제안은 대각(머묾)으로 접는다 = '거절'과 같은 효과."""
    Q = np.zeros((K, K))
    for x in range(K):
        for y in (x - 1, x + 1):
            if 0 <= y < K:
                Q[x, y] += 0.5
            else:
                Q[x, x] += 0.5
    return Q

def time_reversal(P, pi):
    """시간 반전 사슬 p̂(x,y) = π(y) p(y,x) / π(x)  (행렬로 D_π^{-1} Pᵀ D_π)."""
    return (pi[:, None] * P).T / pi[:, None]

def exact_acf(P, pi, f, k_max):
    """정상 상태 자기상관 ρ_k = Σ_x π(x) f̄(x) (P^k f̄)(x) / Var_π f,  k = 0..k_max."""
    fbar = f - (pi * f).sum()
    var = (pi * fbar**2).sum()
    rho, g = np.empty(k_max + 1), fbar.copy()        # g = P^k f̄ 를 한 번에 한 걸음씩 갱신
    for k in range(k_max + 1):
        rho[k] = (pi * fbar * g).sum() / var
        g = P @ g
    return rho

def acf(x, k_max):
    """1차원 표본열의 경험 자기상관 (lag 0..k_max): 평균을 빼고 lag-0 으로 나눈다."""
    xc = x - x.mean()
    c0 = (xc * xc).mean()
    return np.array([1.0] + [(xc[:-k] * xc[k:]).mean() / c0 for k in range(1, k_max + 1)])

def mh_walk(target, n_steps, x0, rng):
    """±1 제안 Metropolis 를 명시적으로 돌린다. 경계 밖 제안은 거절. (path, accepted) 반환."""
    K = len(target)
    path = np.empty(n_steps + 1, dtype=int)
    accepted = np.zeros(n_steps, dtype=bool)
    steps, u = rng.choice([-1, 1], size=n_steps), rng.random(n_steps)
    path[0] = x = x0
    for k in range(n_steps):
        y = x + steps[k]
        if 0 <= y < K and u[k] < target[y] / target[x]:   # α = min(1, π(y)/π(x)); u<α 이면 수락
            x, accepted[k] = y, True
        path[k + 1] = x
    return path, accepted

### 4.1 가역성 검사 — 흐름 행렬 $F=D_\pi P$의 대칭성

(a) 공 20개 Ehrenfest 사슬과 $\pi=\text{Bin}(20,1/2)$: $F$가 대칭이어야 한다. (b) 3-순환 편향 사슬
$P_3=\begin{pmatrix}0&0.9&0.1\\0.1&0&0.9\\0.9&0.1&0\end{pmatrix}$: 이중확률행렬이므로 정상분포는 균등이지만, 시계방향($0\to1\to2\to0$)으로 0.9씩 도는 순환 흐름이 있어 $F$는 비대칭이어야 한다. 시간 반전 $\hat P=D_\pi^{-1}P_3^\top D_\pi=P_3^\top$ (균등 $\pi$)는 반시계로 도는 사슬, 고유값은 복소수.

In [ ]:
states = np.arange(21)
E = ehrenfest_matrix(20)
pi_E = binom.pmf(states, 20, 0.5)
F_E = pi_E[:, None] * E                       # F(x,y) = π(x) p(x,y)
print("Ehrenfest(20): πP = π ?", np.allclose(pi_E @ E, pi_E), "| F symmetric ?", np.allclose(F_E, F_E.T))
assert np.allclose(F_E, F_E.T)                # Ehrenfest detailed balance

P3 = np.array([[0.0, 0.9, 0.1],
               [0.1, 0.0, 0.9],
               [0.9, 0.1, 0.0]])
pi_3 = stationary(P3)
F_3 = pi_3[:, None] * P3
P3_hat = time_reversal(P3, pi_3)
print("biased 3-cycle: π =", np.round(pi_3, 4), "| F symmetric ?", np.allclose(F_3, F_3.T))
print("time reversal P̂ = P3ᵀ ?", np.allclose(P3_hat, P3.T))
print("P̂ =\n", np.round(P3_hat, 2))
print("eigenvalues of P3:", np.round(np.linalg.eigvals(P3), 4))
assert not np.allclose(F_3, F_3.T) and np.allclose(P3_hat, P3.T)   # not reversible; reversal is Pᵀ

두 흐름 행렬을 나란히 그리면 "세부균형 = 대칭 그림"이 눈에 들어온다. Ehrenfest는 대각선 양옆으로 대칭인 띠, 3-순환은 대각선 한쪽(시계방향)만 진하다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2), gridspec_kw={"width_ratios": [2, 1]})
im0 = axes[0].imshow(F_E, cmap="Blues", origin="lower")
axes[0].set(title="Ehrenfest(20): F = F$^T$ (reversible)", xlabel="y", ylabel="x")
fig.colorbar(im0, ax=axes[0], fraction=0.046)
im1 = axes[1].imshow(F_3, cmap="Blues", origin="lower", vmin=0, vmax=F_3.max())
axes[1].set(title="Biased 3-cycle: F $\\neq$ F$^T$", xlabel="y", ylabel="x", xticks=[0, 1, 2], yticks=[0, 1, 2])
for (i, j), v in np.ndenumerate(F_3):
    axes[1].text(j, i, f"{v:.2f}", ha="center", va="center", color="k" if v < 0.2 else "w")
for ax in axes:
    ax.grid(False)
fig.suptitle("Detailed balance = symmetric flow matrix F(x, y) = $\\pi$(x) p(x, y)")
fig.tight_layout()

### 4.2 Metropolis 사슬 구성과 정확한 스펙트럼·자기상관

목표 `target = Bin(20, 0.3)`, 제안 `Q_prop = proposal_pm1(21)`. `metropolis_chain`은 $y\ne x$ 항에만 $\alpha$를 곱하고 나머지를 대각으로 모으므로, 제안행렬의 대각에 접어 둔 "경계 밖" 확률은 그대로 머묾이 된다 — 거절과 같은 효과.

In [ ]:
K = 21
target = binom.pmf(states, 20, 0.3)
mean_exact = (target * states).sum()
var_exact = (target * (states - mean_exact) ** 2).sum()
Q_prop = proposal_pm1(K)
P_mh = metropolis_chain(target, Q_prop)
F_mh = target[:, None] * P_mh
pi_mh = stationary(P_mh)
print(f"target mean = {mean_exact:.4f}, var = {var_exact:.4f}")
print("row-stochastic:", is_stochastic(P_mh), "| stationary == target:", np.allclose(pi_mh, target),
      f"(max |diff| = {np.abs(pi_mh - target).max():.1e}) | F symmetric:", np.allclose(F_mh, F_mh.T))
assert is_stochastic(P_mh) and np.allclose(pi_mh, target) and np.allclose(F_mh, F_mh.T)

lam_mh = np.sort(np.linalg.eigvals(P_mh).real)[::-1]        # 가역 ⇒ 실수 (허수부 ~1e-16 은 버림)
accept_exact = 1.0 - (target * np.diag(P_mh)).sum()          # 정상 상태 수락률 = 1 - P_π(머묾)
print("eigenvalues (top 5):", np.round(lam_mh[:5], 6), "| min:", np.round(lam_mh[-1], 6))
print(f"λ2 = {lam_mh[1]:.6f}, spectral bound (1+λ2)/(1-λ2) = {(1 + lam_mh[1]) / (1 - lam_mh[1]):.4f}")
print(f"stationary acceptance rate = {accept_exact:.6f}")

In [ ]:
fig, ax = plt.subplots()
ax.bar(states, target, width=0.8, alpha=0.5, label="target Bin(20, 0.3)")
ax.plot(states, pi_mh, "o", color="C3", ms=5, label="stationary(P_mh)")
ax.set(xlabel="x", ylabel="probability", title="Metropolis chain reproduces the target exactly")
ax.legend();

정확한 자기상관은 행렬 거듭제곱으로 계산한다: $\rho_k=\sum_x\pi(x)\bar f(x)(P^k\bar f)(x)/\sigma_f^2$. $\rho_k\le\lambda_2^k$이므로 $k=500$까지 더하면 꼬리는 $0.908^{500}\approx10^{-21}$, 무시된다. 길이 $L=800$ 사슬의 유한 보정값 $\tau_L=1+2\sum_{k<L}(1-k/L)\rho_k$도 함께 구한다(4.4절의 정확값).

In [ ]:
K_MAX_ACF, L_SEG = 500, N_STEPS - BURN_IN                    # L_SEG = 800
f = states.astype(float)
rho_exact = exact_acf(P_mh, target, f, max(K_MAX_ACF, L_SEG))
tau_exact = 1.0 + 2.0 * rho_exact[1:K_MAX_ACF + 1].sum()
tau_bound = (1 + lam_mh[1]) / (1 - lam_mh[1])
k_L = np.arange(1, L_SEG)
tau_L = 1.0 + 2.0 * ((1 - k_L / L_SEG) * rho_exact[k_L]).sum()
sd_chain_mean = np.sqrt(var_exact * tau_L / L_SEG)
print("ρ_1..ρ_5 =", np.round(rho_exact[1:6], 6))
print(f"τ = {tau_exact:.4f}  (bound {tau_bound:.4f}),  τ_L(L={L_SEG}) = {tau_L:.4f}")
print(f"n_eff for n = 1e5: {1e5 / tau_exact:.0f}  |  sd of an L={L_SEG} chain mean: {sd_chain_mean:.4f}")

### 4.3 번인 — 출발점의 기억은 $\lambda_2^n$ 속도로 사라진다

꼬리 $X_0=20$에서 출발한 경로 `N_PATHS`개의 평균을 정확 궤적 $E[X_n\mid X_0=20]=(P^n)_{20,\cdot}\cdot x$와 겹친다. $\lambda_2=0.908$이면 $0.908^{50}\approx0.008$, $0.908^{100}\approx6\times10^{-5}$ — 100걸음이면 평균이 6.004까지 온다. 번인 길이 200(이 사슬에서 $\lambda_2^{200}\approx4\times10^{-9}$)은 넉넉하다.

In [ ]:
n_burn_plot = 100
paths = simulate_chain(P_mh, 20, n_burn_plot, rng, N_PATHS)
t = np.arange(n_burn_plot + 1)
mean_sim = paths.mean(0)
mean_traj = np.array([matrix_power(P_mh, n)[20] @ f for n in t])
print("exact E[X_n | X_0=20]:", {n: round(float(mean_traj[n]), 4) for n in (5, 10, 20, 30, 50, 100)})

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
plot_paths(t, paths, ax=axes[0], n_show=10, alpha=0.6)
axes[0].set(title="10 Metropolis paths from $x_0$ = 20", xlabel="n", ylabel="$X_n$")
axes[1].plot(t, mean_sim, lw=2, label=f"mean of {N_PATHS} paths")
axes[1].plot(t, mean_traj, "--", color="C3", label="exact $E[X_n | X_0 = 20]$")
C_asym = (mean_traj[-1] - mean_exact) / lam_mh[1] ** n_burn_plot     # 큰 n 에서 E[X_n] - 6 ≈ C λ2^n
axes[1].plot(t, mean_exact + C_asym * lam_mh[1] ** t, ":", color="gray", label=f"asymptote $6 + C\\,\\lambda_2^n$ (C = {C_asym:.1f})")
axes[1].axhline(mean_exact, color="k", lw=0.8)
axes[1].set(title=f"Burn-in: memory of the start decays at rate $\\lambda_2$ = {lam_mh[1]:.3f}", xlabel="n", ylabel="$E[X_n]$",
            ylim=(5, 21))                                             # 점근선은 n ≳ 40 에서만 의미 있음
axes[1].legend()
fig.tight_layout()

### 4.4 독립 사슬 여러 개 — 시간 평균의 분산에 $\tau$가 곱해진다

`mh_walk`로 사슬 `N_CHAINS`개를 각각 `N_STEPS`걸음 돌린다. 출발점은 $\pi$에서 뽑는다(실전에서는 불가능하다 — 그게 되면 MCMC가 필요 없다; 여기서는 정리 4의 $\tau_L$ 공식이 정확히 성립하게 하려는 것이다). 그래도 실전 절차대로 앞 `BURN_IN`걸음을 버리고 남은 $L=800$개의 (i) 평균 $m_i$, (ii) 수락 비율 $a_i$를 기록한다. **사슬 간에는 독립**이므로 $m_i$, $a_i$에 `mc_estimate`를 쓸 수 있고, $m_i$의 표본분산으로부터 $\hat\tau=\operatorname{Var}(m)\cdot L/\sigma_f^2$를 얻는다.

In [ ]:
chain_means = np.empty(N_CHAINS)
accept_frac = np.empty(N_CHAINS)
for i in range(N_CHAINS):
    x0 = rng.choice(K, p=target)                              # 정상 출발
    path, accepted = mh_walk(target, N_STEPS, x0, rng)
    chain_means[i] = path[BURN_IN + 1:].mean()                # 번인 뒤 L = 800 개
    accept_frac[i] = accepted.mean()
est_mean = mc_estimate(chain_means)
est_accept = mc_estimate(accept_frac)
tau_hat = chain_means.var(ddof=1) * L_SEG / var_exact
se_tau_hat = tau_L * np.sqrt(2 / (N_CHAINS - 1))              # 정규근사: Var(s²) ≈ 2σ⁴/(n-1)
print(f"chain means : {est_mean}   (exact {mean_exact:.4f})")
print(f"accept rate : {est_accept}   (exact {accept_exact:.6f})")
print(f"sd of chain means = {chain_means.std(ddof=1):.4f} vs sqrt(σ²τ_L/L) = {sd_chain_mean:.4f}"
      f"  |  naive sqrt(σ²/L) = {np.sqrt(var_exact / L_SEG):.4f}")
print(f"tau_hat = {tau_hat:.2f} ± {se_tau_hat:.2f}   (exact τ_L = {tau_L:.4f})")

길이 800 사슬의 평균은 독립 표본 800개의 평균보다 $\sqrt{\tau_L}\approx4.5$배 더 퍼져 있다. 히스토그램 위에 두 정규곡선(순진한 SE와 $\tau$ 보정 SE)을 겹쳐 보자.

In [ ]:
fig, ax = plt.subplots()
ax.hist(chain_means, bins=25, density=True, alpha=0.5, label=f"{N_CHAINS} chain means (L = {L_SEG})")
xx = np.linspace(4.5, 7.5, 300)
ax.plot(xx, norm.pdf(xx, mean_exact, sd_chain_mean), color="C3", lw=2,
        label=f"N(6, $\\sigma^2\\tau_L/L$), sd = {sd_chain_mean:.3f}")
ax.plot(xx, norm.pdf(xx, mean_exact, np.sqrt(var_exact / L_SEG)), "--", color="gray",
        label=f"naive N(6, $\\sigma^2/L$), sd = {np.sqrt(var_exact / L_SEG):.3f}")
ax.axvline(mean_exact, color="k", lw=0.8)
ax.set(xlabel="chain mean", ylabel="density", title=f"Chain means: variance inflated by $\\tau \\approx$ {tau_L:.0f}")
ax.legend();

### 4.5 자기상관 함수 — 정상 출발 쌍과 긴 사슬 하나

(a) `simulate_chain`에 **확률벡터** `target`을 `x0`로 넘기면 정상 출발 $(X_0,X_1)$ 쌍을 독립으로 `N_PAIRS`개 뽑는다. $\rho_1=E[\bar f(X_0)\bar f(X_1)]/\sigma_f^2$이므로 `prod = (X0-6)(X1-6)/4.2`의 평균이 $\rho_1$의 불편추정량이고, 쌍이 독립이라 `mc_estimate`의 SE가 정직하다. (b) 긴 사슬 하나의 경험 ACF를 정확 $\rho_k$와 겹친다 — 경험 ACF의 잡음은 $\sim\sqrt{\tau/n}$ 규모라 $\rho_k$가 그 아래로 내려가면(큰 $k$) 마커가 흩어진다.

In [ ]:
pairs = simulate_chain(P_mh, target, 1, rng, N_PAIRS)          # x0 = 확률벡터 → 정상 출발
prod = (pairs[:, 0] - mean_exact) * (pairs[:, 1] - mean_exact) / var_exact
est_rho1 = mc_estimate(prod)
print(f"rho_1 from {N_PAIRS} stationary pairs: {est_rho1}   (exact {rho_exact[1]:.6f})")

K_MAX = 60
long_chain = simulate_chain(P_mh, target, N_LONG, rng)[0]
rho_emp = acf(long_chain.astype(float), K_MAX)
print(f"one chain of length {N_LONG}: empirical rho_1..3 = {np.round(rho_emp[1:4], 4)}, "
      f"n_eff = n/τ ≈ {N_LONG / tau_exact:.0f}")

fig, ax = plt.subplots()
kk = np.arange(K_MAX + 1)
ax.semilogy(kk, rho_exact[kk], color="C3", lw=2, label="exact $\\rho_k$")
ax.semilogy(kk, lam_mh[1] ** kk, "--", color="gray", label="$\\lambda_2^k$")
ax.semilogy(kk, rho_emp, "o", ms=4, alpha=0.7, label=f"empirical (one chain, n = {N_LONG})")
ax.axhline(np.sqrt(tau_exact / N_LONG), color="C0", ls=":", label="noise level $\\sim\\sqrt{\\tau/n}$")
ax.set(xlabel="lag k", ylabel="$\\rho_k$", ylim=(1e-3, 1.5),
       title=f"Autocorrelation of $X_n$ under the Metropolis chain: $\\tau$ = {tau_exact:.1f}")
ax.legend();

## 5. Compare — 예측 · 시뮬레이션 · 닫힌 형식 비교

In [ ]:
rows = [
    {"name": "mean_target (chain means)", "predicted": predictions["mean_target"],
     "estimate": est_mean, "exact": mean_exact},
    {"name": "accept_rate", "predicted": predictions["accept_rate"],
     "estimate": est_accept, "exact": accept_exact},
    {"name": "rho1", "predicted": predictions["rho1"],
     "estimate": est_rho1, "exact": rho_exact[1]},
    {"name": "tau_hat (vs tau_L)", "predicted": predictions["tau_int"],
     "estimate": tau_hat, "se": se_tau_hat, "exact": tau_L},
]
Markdown(compare_table(rows))

- **mean_target**: 어긋나면 사슬의 정상분포가 $\pi$가 아니다 — `mh_walk`의 수락 규칙(특히 경계 처리)이 `P_mh`와 다르다는 뜻.
- **accept_rate**: 어긋나면 `mh_walk`의 제안/수락이 `Q_prop`·$\alpha$와 다르거나, 정상 출발이 아니다.
- **rho1**: 어긋나면 `exact_acf`의 공식(정상 상태 공분산)이나 `simulate_chain`의 확률벡터 출발이 잘못됐다.
- **tau_hat**: 어긋나면(SE $\approx2$로 느슨한 검사다) 시간 평균의 분산이 $\sigma^2\tau_L/L$이 아니다 — 정리 4 자체나 사슬 간 독립성이 깨진 것.

예측값 `predictions`는 표에만 나타나며, 아래 검정은 **MC 추정 vs 닫힌 형식**만 본다. 예측 셀에 적은 숫자와 표의 정확값이 다르면 그 이유를 2절에서 찾아 보세요.

In [ ]:
assert abs(lam_mh[1] - 0.907904) < 1e-5 and abs(tau_exact - 20.2214) < 1e-3     # 스펙 교차검증
assert_close(est_mean, mean_exact, k=4, name="mean_target")
assert_close(est_accept, accept_exact, k=4, name="accept_rate")
assert_close(est_rho1, rho_exact[1], k=4, name="rho1")
assert_close(tau_hat, tau_L, se=se_tau_hat, k=4, name="tau_hat")
print("all MC-vs-exact checks passed (k=4)")

## 6. 연습문제

### E1 계산 — 세부균형·시간 반전 손계산

(a) 공 3개 Ehrenfest 사슬($p(k,k+1)=(3-k)/3$, $p(k,k-1)=k/3$)에서 $\pi=(1,3,3,1)/8$이 세부균형을 만족함을 모든 인접 쌍 $(0,1),(1,2),(2,3)$에 대해 손으로 확인하시오.
(b) 3-순환 편향 사슬 $P_3$의 시간 반전 $\hat P$를 공식 $\hat p(x,y)=\pi(y)p(y,x)/\pi(x)$로 구하시오.
(c) $P=\begin{pmatrix}0.5&0.5\\0.25&0.75\end{pmatrix}$는 가역인가? 2-상태 사슬은 언제나 가역인가?

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

(a) 흐름 $F(k,k+1)=\pi(k)p(k,k+1)$과 $F(k+1,k)$:
$\pi(0)p(0,1)=\frac18\cdot1=\frac18$, $\pi(1)p(1,0)=\frac38\cdot\frac13=\frac18$ ✓;
$\pi(1)p(1,2)=\frac38\cdot\frac23=\frac14$, $\pi(2)p(2,1)=\frac38\cdot\frac23=\frac14$ ✓;
$\pi(2)p(2,3)=\frac38\cdot\frac13=\frac18$, $\pi(3)p(3,2)=\frac18\cdot1=\frac18$ ✓.
비인접 쌍은 양변이 0. 따라서 세부균형, 정리 1로 $\pi P=\pi$ — 01d 정리 2의 $n=3$ 경우.

(b) $\pi$가 균등이므로 $\hat p(x,y)=p(y,x)$, 즉 $\hat P=P_3^\top=\begin{pmatrix}0&0.1&0.9\\0.9&0&0.1\\0.1&0.9&0\end{pmatrix}$: 반시계 방향($0\to2\to1\to0$)으로 0.9씩 도는 사슬. 원래 사슬과 다르므로 비가역.

(c) $\pi P=\pi$: $\pi(0)=0.5\pi(0)+0.25\pi(1)\Rightarrow\pi(1)=2\pi(0)$, $\pi=(1/3,2/3)$. 세부균형은 쌍 $(0,1)$ 하나뿐: $\pi(0)p(0,1)=\frac13\cdot\frac12=\frac16=\frac23\cdot\frac14=\pi(1)p(1,0)$ ✓ — 가역. **모든 2-상태 사슬은 가역**이다: 정상성 $\pi(0)=\pi(0)p(0,0)+\pi(1)p(1,0)$에서 $\pi(0)(1-p(0,0))=\pi(0)p(0,1)=\pi(1)p(1,0)$이 바로 세부균형. 순환 흐름이 생기려면 상태가 3개 이상 필요하다(Kolmogorov 순환 판정, Ross 12e §4.8 Thm 4.2).

```python
E3 = ehrenfest_matrix(3); pi3 = np.array([1, 3, 3, 1]) / 8
print([pi3[k] * E3[k, k + 1] for k in range(3)], np.allclose(pi3[:, None] * E3, (pi3[:, None] * E3).T))
print(time_reversal(P3, pi_3))
P2 = np.array([[0.5, 0.5], [0.25, 0.75]]); pi2 = stationary(P2)
print(pi2, np.allclose(pi2[:, None] * P2, (pi2[:, None] * P2).T))      # [1/3, 2/3] True
```

</details>

### E2 유도 후 시뮬레이션 검증 — 독립 제안 (independence sampler)

제안이 현재 상태에 무관할 때, $q(x,y)=g(y)$, MH 수락확률이 $\min\big(1,\frac{\pi(y)g(x)}{\pi(x)g(y)}\big)$임을 유도하시오. 목표 $\text{Bin}(20,0.3)$, $g$ = 균등$\{0,\dots,20\}$으로 `P_ind = metropolis_chain(target, np.ones((21, 21)) / 21)`을 만들어 정상분포가 `target`임을 확인하고, $\lambda_2$와 정상 상태 수락률(= 제안 $y\ne x$가 수락될 확률)을 **정확히** 계산한 뒤, 정상 출발 쌍 `N_PAIRS`개에서 '이동한 비율' `mc_proportion(X1 != X0)`으로 검증하시오. $\pm1$ 걷기(수락률 81%, $\lambda_2=0.908$)와 비교해 어느 쪽이 더 효율적인가?

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

**유도.** 일반식에 $q(x,y)=g(y)$, $q(y,x)=g(x)$를 넣으면 $\alpha(x,y)=\min\big(1,\frac{\pi(y)g(x)}{\pi(x)g(y)}\big)$: 비율 $\pi/g$("중요도 가중치")가 현재보다 큰 곳으로는 항상, 작은 곳으로는 가중치 비율만큼 간다. $g$가 균등이면 $g(x)/g(y)=1$이라 $\alpha=\min(1,\pi(y)/\pi(x))$ — 형태는 Metropolis와 같지만 제안이 전역이다.

**정확값.** 균등 제안은 $y=x$도 확률 $1/21$로 제안하는데 이는 항상 "수락"되지만 이동이 아니다. 문제의 정의(제안 $y\ne x$가 수락될 확률)는 $\sum_x\pi(x)\sum_{y\ne x}\frac1{21}\min(1,\pi(y)/\pi(x))=1-\sum_x\pi(x)p_{\rm ind}(x,x)$이고 이는 곧 정상 상태에서 $X_1\ne X_0$일 확률이므로 `mc_proportion(X1 != X0)`과 같다: $0.265729$ (SE $\approx\sqrt{0.27\cdot0.73/N_{\rm PAIRS}}$). $\lambda_2=0.751517$.

**비교.** 수락률은 독립 제안이 훨씬 낮지만($27\%$ vs $81\%$) $\lambda_2$가 작아 $\tau$ 상계 $(1+\lambda_2)/(1-\lambda_2)=7.05$ vs $20.7$ — 독립 제안이 약 3배 효율적이다. 수락률은 효율의 척도가 아니다(함정 2). 단, 독립 제안은 $g$가 $\pi$의 꼬리를 덮지 못하면(여기서는 균등이라 문제없음) 특정 상태에 오래 갇혀 최악이 된다.

```python
P_ind = metropolis_chain(target, np.ones((K, K)) / K)
print("stationary == target:", np.allclose(stationary(P_ind), target))
lam_ind = np.sort(np.linalg.eigvals(P_ind).real)[::-1]
accept_ind = 1.0 - (target * np.diag(P_ind)).sum()
print(f"λ2 = {lam_ind[1]:.6f} (bound {(1 + lam_ind[1]) / (1 - lam_ind[1]):.2f}), acceptance = {accept_ind:.6f}")
pairs_ind = simulate_chain(P_ind, target, 1, rng, N_PAIRS)
est_move = mc_proportion(pairs_ind[:, 1] != pairs_ind[:, 0])
print("moved fraction:", est_move); assert_close(est_move, accept_ind, k=4, name="indep. sampler acceptance")
```

</details>

### E3 가정을 깨보기 — 쌍봉 목표에서 '사실상' 기약이 아닌 사슬

기약성을 사실상 깨자. 쌍봉 목표 $\pi(x)\propto e^{-(x-5)^2/4.5}+e^{-(x-15)^2/4.5}$ on $\{0,\dots,20\}$(평균 10)에 $\pm1$ 제안 MH를 적용하시오. $\lambda_2$를 계산하고, $x_0=5$에서 2000걸음 사슬을 몇 개(예: 8개) 돌려 표본평균이 10 근처인지 보시오. 정확값 $E[X_{2000}\mid X_0=5]$와 $P(X_{2000}>10\mid X_0=5)$도 행렬 거듭제곱으로 구하시오. 무엇이 잘못됐고 어떻게 고치는가?

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

$\lambda_2=0.999229$, $1-\lambda_2=7.7\times10^{-4}$ ⇒ $\tau$ 상계 $(1+\lambda_2)/(1-\lambda_2)\approx2594$. 두 봉우리 사이의 골 $\pi(10)/\pi(5)\approx0.008$을 $\pm1$ 걸음으로 넘으려면 수락률이 매우 낮은 오르막 5걸음을 연속으로 성공해야 하므로 봉우리 사이 이동에 평균 $\sim10^3$걸음이 걸린다. 2000걸음 사슬은 대개 한 봉우리에만 머물러 표본평균이 $\approx5$ 또는 $\approx15$이다. 정확히는 $E[X_{2000}\mid X_0=5]=8.92$(아직 10이 아님), $P(X_{2000}>10\mid X_0=5)=0.39$(반쯤만 건너감). 사슬은 **기약**이고 정리 2의 가정을 모두 만족하지만 $\lambda_2\approx1$이라 실용적으로 갇힌다 — 수렴 정리는 "언젠가"를 말할 뿐이다. 번인을 2000으로 늘려도 $\tau\sim10^3$이라 유효 표본이 거의 없다.

**처방.** (i) 큰 점프 제안(균등 $\pm1,\dots,\pm10$)이나 독립 제안(E2)으로 골을 한 번에 건너뛴다; (ii) 담금질(tempering: $\pi^{1/T}$로 골을 얕게 한 사슬과 교환); (iii) 두 봉우리에서 출발한 사슬 여러 개의 평균이 다르면 수렴 실패 — Gelman–Rubin 진단. 핵심은 $\lambda_2$가 사슬의 실질 속도를 정한다는 것.

```python
target_bi = np.exp(-(states - 5) ** 2 / 4.5) + np.exp(-(states - 15) ** 2 / 4.5)
target_bi /= target_bi.sum()
P_bi = metropolis_chain(target_bi, Q_prop)
lam_bi = np.sort(np.linalg.eigvals(P_bi).real)[::-1]
print(f"mean = {(target_bi * states).sum():.4f}, λ2 = {lam_bi[1]:.6f}, bound = {(1 + lam_bi[1]) / (1 - lam_bi[1]):.0f}")
d2000 = matrix_power(P_bi, 2000)[5]
print(f"E[X_2000 | X_0=5] = {d2000 @ f:.3f},  P(X_2000 > 10 | X_0=5) = {d2000[11:].sum():.3f}")
runs = simulate_chain(P_bi, 5, 2000, rng, 8)
print("sample means of 8 chains of length 2000 from x0=5:", np.round(runs[:, 1:].mean(1), 2))
# 처방 (i): ±1..±10 균등 제안
Q_big = np.zeros((K, K))
for x in range(K):
    for d in range(1, 11):
        for y in (x - d, x + d):
            if 0 <= y < K: Q_big[x, y] += 1 / 20
            else: Q_big[x, x] += 1 / 20
lam_big = np.sort(np.linalg.eigvals(metropolis_chain(target_bi, Q_big)).real)[::-1]
print(f"with ±1..±10 proposals: λ2 = {lam_big[1]:.4f}, bound = {(1 + lam_big[1]) / (1 - lam_big[1]):.1f}")
```

</details>

## 7. 정리

1. **세부균형** $\pi(x)p(x,y)=\pi(y)p(y,x)$ ⇒ 정상분포(정리 1). 검사는 흐름 행렬 $F=D_\pi P$의 대칭성. 정상이어도 순환 흐름이 있으면 비가역(3-순환 편향 사슬).
2. **시간 반전** $\hat P=D_\pi^{-1}P^\top D_\pi$; 가역 $\iff\hat P=P$; 가역이면 $D_\pi^{1/2}PD_\pi^{-1/2}$가 대칭이라 **실수 스펙트럼**(정리 3) — 01e 스펙트럼 상계의 전제.
3. **Metropolis-Hastings**: 제안 $q$ + 수락 $\min\big(1,\frac{\pi(y)q(y,x)}{\pi(x)q(x,y)}\big)$ ⇒ 세부균형 자동(정리 2), 정규화 상수 불필요. `stationary(P_mh) == target`으로 확인했다.
4. **MCMC의 비용은 자기상관**: $\operatorname{Var}(\bar f_n)\approx\sigma^2\tau/n$, $\tau=1+2\sum\rho_k\le\frac{1+\lambda_2}{1-\lambda_2}$(정리 4). $\pm1$ 제안: $\lambda_2=0.908$, $\tau=20.2$, 수락률 81%; 독립 제안: $\lambda_2=0.75$, 수락률 27%가 3배 더 효율적.
5. **쌍봉 목표**에서 $\lambda_2=0.9992$: 기약이지만 실용적으로 갇힘 — 번인·표본 수를 늘려도 $\tau$가 크면 소용없다. 제안을 바꿔 $\lambda_2$를 낮추는 것이 답.

**trap 카드.** Q: MCMC 표본 $n=10^5$개의 평균에 대한 표준오차를 `np.std(x)/np.sqrt(n)`으로 계산해도 되는가?
→ A: 안 된다. 표본이 자기상관되어 실제 분산은 $\sigma^2\tau/n$ ($\tau\approx20$)이므로 SE가 $\sqrt{20}\approx4.5$배 과소평가된다. 배치 평균, 독립 사슬 여러 개(4.4절), 또는 $\tau$ 추정으로 보정한다. 유효 표본은 $n/\tau\approx5000$개뿐.

**다음 노트북: 01g (분기 과정).** 지금까지의 사슬은 유한 상태공간이었다. 개체 수가 무한히 자랄 수 있는 첫 무한 상태 사슬 — 골턴-왓슨 분기 과정 — 에서 '흡수(소멸) 확률'을 pgf의 고정점으로 구한다(01c의 흡수 + 00c의 pgf).

`log/progress.md` 한 줄 템플릿:
`- YYYY-MM-DD 01f 가역성·MCMC — 예측 적중 _/9, compare 판정 ok _/4, E1 _ E2 _ E3 _, 막힌 곳: ___`